# Production test v1 — one arm, end to end, on an independent file

The run described by [`docs/PLAN_prod_test_v0.md`](../docs/PLAN_prod_test_v0.md): the
SOTA AR arm (`ar_junipr_v4` + `lundnet`) trained at `n_bins: 30` on an
**asymmetric-floor** PYTHIA file, assessed here on a **different file from a different
seed**. Three things this settles that no previous result could:

1. the aux conditioning A/B now runs on a sample where the secondary-plane features
   are not constant-zero four times in five (`SoftDrop:ktFloorSec = 0.2`);
2. the leading-emission metric is no longer quantisation-limited (cell 0.6 → 0.2);
3. there is an actual **test split** — an independent file, not a tenth of the
   training one.

It is a **production test of the pipeline**, not a model ranking. One arm, one seed
for the headline; the aux ablation beside it carries a two-seed band, which is the
minimum that can conclude anything at all.

Everything structural is read off the checkpoint snapshot — nothing about the
architecture, the geometry or the aux columns is retyped here.

## What v1 changes

**Nothing about the assessment**: the same tiers, the same estimators, the same
acceptance criterion, the same asserts. This is
[`prod_test_v0.ipynb`](prod_test_v0.ipynb) with the decode plumbing of
[`docs/PLAN_prod_test_speedup.md`](../docs/PLAN_prod_test_speedup.md) applied — **one**
shared sampling pass instead of four, batched coordinate draws, a batched `length_pmf`,
and a thread cap — plus one bug the plan did not know about: §2b/§2c rebuilt their
256-jet dataset **once per jet** (`MatchedLundDataset(chunk, ...)[k]` inside the
comprehension), which is O(B²) and cost ~60 min on its own, more than everything the
plan targeted. Together that takes the notebook from ~109 min to ~6 min for the same
numbers. Three consequences a reader has to know:

* **sharing the draws changes nothing at all** — not "nothing within noise". Every
  cell-level number came back bit-identical to the v0 artifact beside this checkpoint
  (§9 prints the comparison): `dlund_*`, `coverage_68`, the SBC χ², the multiplicity
  bias, the occupancy, TARP. That is not luck. v0 re-seeded with `seed_everything(SEED)`
  before each of its four sampling passes and walked the same tier in the same order, so
  the four passes were drawing *the same draws*; sharing them is an exact refactor, and
  the four passes were paying three times over for a coincidence nobody had checked.
* **the coordinate rows do move**, and only they: `sample_coordinates_many` consumes the
  RNG in a different order by design, so `dlund_*_cont` shifted +0.9% here, and
  `collect()`'s cell rows shifted −0.2% with it because its loop interleaves coordinate
  draws between jets. `METRICS["run"]["shared_draws"]` records the regime either way.
* §6's `length_pmfs` is batched and therefore **bit-identical**, not merely close: no
  RNG is involved, and the notebook asserts it against the batch-1 path rather than
  claiming it.

`prod_test_v0.ipynb` stays in the tree unchanged — it is the record of how the v0
artifact beside the checkpoint was produced.


## 0. Parameters


In [ ]:
# --- inputs -----------------------------------------------------------------
CKPT_PATH = None          # repo-relative; None -> newest runs/prod_test_v0/*/best.ckpt
ROOT_PATH = "data/jet_aux_asym_test.root"      # the INDEPENDENT file (seed 2)
TRAIN_ROOT_PATH = "data/jet_aux_asym.root"     # seed 1; used for §2 agreement + the
#                   frozen tau / (T, tilt) fits, which must NOT see the test file
NTUPLE_NAME = "Jets"

# The aux ablation (check 2). None -> auto-discover under runs/prod_test_v0/ablation/.
# A two-by-two (aux on/off x seed 0/1) is the minimum that can conclude: the previous
# A/B failed because its -0.029 nat delta WAS the 0.029 seed spread.
ABLATION_ROOT = "runs/prod_test_v0/ablation"

# --- jet tiers --------------------------------------------------------------
# Four NESTED tiers over ONE frozen shuffled index list, so every comparison is paired
# and the cost per jet (which spans four orders of magnitude) stays bounded.
N_POP = None      # None -> every jet in the file (NLL, q(0|x), occupancy, marginals)
N_PIT = 5000      # per chunk; `coordinate_pits` collates every jet into ONE padded
N_PIT_CHUNKS = 4  # batch (eval/calibration.py), so 97k at once is multiple GB and OOMs.
#                   The 4 disjoint chunks double as an MC-error estimate.
N_SAMP = 2000     # run_calibration, run_closure(continuous=False), collect()
N_HEAVY = 300     # run_tarp, run_closure(continuous=True)
POP_BATCH = 256   # chunk size for the batched forward passes

SEED = 1234
DEVICE = "cpu"    # the model is tiny and decoded one jet at a time, so a GPU never
#                   amortises its dispatch overhead. The batched §2/§3 passes are the
#                   exception and are cheap either way.

# torch's default is one thread per core, and at batch 1 that is SLOWER, not faster:
# every op pays an N-way fork/join barrier over a few hundred elements. MEASURED on this
# box (20-core Grace CPU, idle, 2026-08-01), 4 threads vs the default 20:
#     sample_batch(K=200)        49 ms/jet   vs  77   -- the largest stage of this run
#     batched sample_coordinates 22 ms/jet   vs  27
#     run_closure cell, per jet  38 ms/jet   vs  30   -- the one row that PREFERS 20
#   net over the notebook: ~10% faster capped, and it is far larger under load --
#   docs/PLAN_prod_test_speedup.md measured ~2x with four trainings live, which is the
#   condition this notebook is usually run in.
# The optimum is flat from 1 to 4 threads, so 4 is a safe cap rather than a tuned one.
# Measured, not folklore: re-measure on your box before changing it (and note that
# docs/PLAN_prod_test_speedup_mac.md finds a different mechanism and magnitude again).
TORCH_THREADS = 4
import torch  # noqa: E402  -- here, not in §1, so the cap precedes any tensor work

_DEFAULT_THREADS = torch.get_num_threads()
torch.set_num_threads(TORCH_THREADS)
print(f"torch threads: {_DEFAULT_THREADS} (this box's default) -> "
      f"{torch.get_num_threads()}")

# --- inference knobs --------------------------------------------------------
K_DRAWS = 200     # posterior draws per jet
# `surrogate` is a DIFFERENT risk function AND, before the check-6 fix, binned at 10
# while the model decides at 30. Hard-asserted against below: no reported number may
# use it. The empty-tree column is the one observable that swings with the backend
# (pot ~0.2%, surrogate ~57%), so §6 quotes it explicitly.
MBR_BACKEND = "pot"

WRITE_ARTIFACTS = True   # -> <ckpt dir>/prod_test_v1/prod_test_v1_metrics.json

# COST. MEASURED 5.9 min end to end, top to bottom, on this box (20-core Grace CPU,
# idle, 2026-08-01, TORCH_THREADS = 4) — against v0's ~109 min for the same numbers, and
# in truth more: that budget never counted §2b/§2c, which spent ~60 min rebuilding one
# dataset per JET (see the note in §2b). Per cell, from that run, beside v0's budget:
#
#   section                                 v0         v1    what changed
#   §1 ONE shared sampling pass             --     1.43 min  the draws every section reuses
#   §2c aux ablation, 5 arms x 97k jets  ~60 min   1.35 min  ONE dataset per chunk, not B
#   §5 run_closure cell, 2000 jets       7.7 min   1.25 min  draws_by_jet=DRAWS
#   §4 TARP, 300 jets                    1.0 min      23 s   EMD-bound; unchanged
#   §2b held-out NLL, 97k jets              --        20 s   the same O(B^2) fix
#   §5 run_closure continuous, 300      35.9 min      17 s   ONE batched coordinate call
#   §6 (T, tilt) fit on 20k jets            --        17 s   unchanged; now the visible one
#   §5 collect(), 2000 + 300            38.8 min       8 s   shared draws + batched coords
#   §6 fit set + both length_pmfs       13.3 min       7 s   batched n_head, bit-identical
#   §8 support, 300 jets                 4.1 min       7 s   shared draws + batched coords
#   §3 occupancy, 2000 jets              4.1 min      ~0 s   reads the shared draws
#   §4 run_calibration, 2000 jets        4.1 min      ~0 s   draws_by_jet=DRAWS
#   (both file loads, §2, §2b's table, the PIT chunks and §7/§9: 12 s together)
#
# The knobs keep their meaning: K_DRAWS is linear in the shared pass, N_SAMP sets its
# length, N_HEAVY only the continuous rows. The per-jet coordinate loop that dominated v0
# is gone — one `sample_coordinates_many` call per jet costs 22 ms where K per-draw calls
# cost 2.5 s (measured, 115x on that stage alone).


## 1. Load the model and the test file

Every assertion here is a hard one. Each corresponds to a risk in the plan that would
silently void the numbers rather than fail loudly:
a `kt_floor_sec == kt_floor` file makes the whole framing inert; an `AUX == ()`
checkpoint makes it inert *by construction*; a geometry mismatch means the cell
metric is not the one the plan sizes; and two files with different cards measure
covariate shift instead of generalisation.


In [ ]:
import importlib.util
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import torch
from omegaconf import OmegaConf

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from h2p_rsd_junipr.config import decode_params  # noqa: E402
from h2p_rsd_junipr.data.dataset import MatchedLundDataset, collate  # noqa: E402
from h2p_rsd_junipr.data.rntuple import load_rntuple  # noqa: E402
from h2p_rsd_junipr.data.stats import check_multiplicity_support, multiplicity_stats  # noqa: E402
from h2p_rsd_junipr.eval.calibration import (  # noqa: E402
    REGION_LABELS,
    cell_region,
    chi2_crit95,
    coordinate_pits,
    run_calibration,
    wilson_interval,
)
from h2p_rsd_junipr.eval.closure import leading_emission_cell, lund_distance, run_closure  # noqa: E402
from h2p_rsd_junipr.eval.report import inert_decode_keys, plot_calibration, save_metrics  # noqa: E402
from h2p_rsd_junipr.features import node_raw  # noqa: E402
from h2p_rsd_junipr.geometry import Geometry  # noqa: E402
from h2p_rsd_junipr.inference.length import (  # noqa: E402
    empty_gate,
    empty_threshold_for_rate,
    fit_length_recalibration,
    recalibrate_pmf,
)
from h2p_rsd_junipr.models.base import build_model  # noqa: E402
from h2p_rsd_junipr.train.checkpoint import load_for_inference  # noqa: E402
from h2p_rsd_junipr.train.trainer import seed_everything  # noqa: E402

METRICS: dict = {}     # everything quotable accumulates here; written in §9


def rel(p):
    """Repo-relative when it can be, absolute otherwise — a checkpoint kept outside the
    tree is a legitimate thing to point at and must not crash the report."""
    p = Path(p)
    try:
        return p.relative_to(REPO)
    except ValueError:
        return p


def newest_ckpt(root):
    cands = sorted(Path(root).glob("*/best.ckpt"), key=lambda p: p.stat().st_mtime)
    if not cands:
        raise FileNotFoundError(f"no */best.ckpt under {root}")
    return cands[-1]


CKPT = Path(CKPT_PATH) if CKPT_PATH else newest_ckpt(REPO / "runs/prod_test_v0")
if not CKPT.is_absolute():
    CKPT = REPO / CKPT
device = torch.device(DEVICE)
info = load_for_inference(CKPT, map_location=device)
cfg = OmegaConf.create(info["config"])
geom = Geometry.from_config(cfg.geometry)
model = build_model(cfg, geom).to(device)
model.load_state_dict(info["model_state"])
model.eval()

AUX = tuple(model.aux_feature_names)
DECODE = decode_params(cfg)

print(f"checkpoint      {rel(CKPT)}")
print(f"family          {info['model_name']} + {cfg.encoder.name}"
      f"   ({sum(p.numel() for p in model.parameters()) / 1e3:.1f}k params)")
print(f"trained         {info['epoch']} epochs, best val NLL/jet = {info['best_val_nll']:.4f}")
print(f"geometry        n_bins={geom.n_bins} -> {geom.n_cells} cells,"
      f" cell {geom.cell_wu:.2f} x {geom.cell_wv:.2f},"
      f" half_u/half_v = {geom.half_u:.2f}/{geom.half_v:.2f}")
print(f"aux ({len(AUX)})       {list(AUX)}")
print(f"trained on      {cfg.data.path}")


In [ ]:
jets = load_rntuple(str(REPO / ROOT_PATH), NTUPLE_NAME)
assert jets, f"no jets read from {ROOT_PATH}"

prov = {k: jets[0][k] for k in ("z_cut", "beta", "kt_floor", "kt_floor_sec", "generator")}
print("test-file provenance:", prov)

# --- the hard asserts -------------------------------------------------------
# 1. The file really is asymmetric. If the two floors match, the aux columns are
#    groomed exactly like the sequences and the entire reason for this file is gone.
assert prov["kt_floor_sec"] != prov["kt_floor"], (
    f"kt_floor_sec == kt_floor == {prov['kt_floor']}: this is a SYMMETRIC file. The "
    f"asymmetric floor is what makes x_nsec non-degenerate; without it this test has "
    f"no subject."
)
# 2. The checkpoint conditions on aux. The asymmetric floor leaves x/y bit-for-bit
#    unchanged, so with AUX == () the file is inert BY CONSTRUCTION.
assert AUX, (
    "the loaded checkpoint has encoder.aux_features=[]. The asymmetric floor changes "
    "ONLY the aux columns, so an aux-free arm cannot see this file at all — every "
    "number below would be identical on the symmetric one."
)
# 3. The geometry is the one the plan sizes the metric for.
assert geom.n_bins == 30, f"expected n_bins=30, checkpoint says {geom.n_bins}"
# 4. `surrogate` is a different risk function and, at 30 bins, a coarser one.
assert MBR_BACKEND != "surrogate", "no reported number may use the surrogate backend"
# 5. The aux column set must match what the checkpoint was trained with, or the
#    dataset builds a different-width xf and the model silently reads the wrong column.
ds = MatchedLundDataset(jets, geom, AUX)
assert ds[0]["xf"].shape[1] == 5 + len(AUX)

print(f"\n{len(jets)} test jets, dataset width {ds[0]['xf'].shape[1]} = 5 + {len(AUX)} aux")


In [ ]:
# The other half of the disjointness guard: same card, different sample. `check_disjoint`
# already asserted no shared jet; this asserts the two files are the same PHYSICS.
train_jets = load_rntuple(str(REPO / TRAIN_ROOT_PATH), NTUPLE_NAME)
assert train_jets, f"no jets read from {TRAIN_ROOT_PATH}"
train_prov = {k: train_jets[0][k] for k in prov}
assert train_prov == prov, (
    f"train/test grooming provenance differs — {train_prov} vs {prov}. The assessment "
    f"would measure covariate shift, not generalisation."
)
print(f"train file {len(train_jets)} jets, provenance identical to the test file.")

disjoint_path = REPO / "runs/prod_test_v0/disjoint.json"
if disjoint_path.exists():
    DISJOINT = json.loads(disjoint_path.read_text())
    assert DISJOINT["passed"], "scripts/check_disjoint.py FAILED — the seeds collided"
    print(f"seed-collision guard: PASS "
          f"({DISJOINT['overlap']['full']['n_overlap']} shared jets of "
          f"{DISJOINT['overlap']['full']['n_b']} compared)")
else:
    DISJOINT = None
    print("WARNING: runs/prod_test_v0/disjoint.json absent — run "
          "`python scripts/check_disjoint.py data/jet_aux_asym.root "
          "data/jet_aux_asym_test.root` before quoting anything below.")

METRICS["run"] = {
    "checkpoint": str(rel(CKPT)),
    "model": info["model_name"], "encoder": str(cfg.encoder.name),
    "epochs": info["epoch"], "best_val_nll_train_file": info["best_val_nll"],
    "n_bins": geom.n_bins, "n_cells": geom.n_cells,
    "aux_features": list(AUX),
    "train_path": str(cfg.data.path), "test_path": ROOT_PATH,
    "n_test_jets": len(jets), "n_train_jets": len(train_jets),
    "provenance": {k: (float(v) if k != "generator" else v) for k, v in prov.items()},
    "mbr_backend": MBR_BACKEND, "K_draws": K_DRAWS, "seed": SEED,
    # The v1 plumbing, recorded because it decides how this artifact may be compared to
    # a v0 one: ONE sampling pass feeds §3/§4/§5/§8, and the coordinate draws are batched
    # per jet. The first is an exact refactor (v0 re-seeded before each pass, so all four
    # drew the same draws); the second reorders RNG consumption, so the `*_cont` rows —
    # and only those — are not bit-comparable across the two.
    "notebook": "prod_test_v1", "shared_draws": True,
    "torch_threads": int(TORCH_THREADS),
    "disjoint_check": DISJOINT,
}


In [ ]:
# ONE frozen shuffled index list; every tier is a prefix of it, so the tiers are nested
# and every comparison between them is paired. Shuffled because the file is written in
# event order, and a prefix of that is a prefix of the pT-hat evolution, not a sample.
rng = np.random.default_rng(SEED)
ORDER = rng.permutation(len(jets))
POP = ORDER if N_POP is None else ORDER[:N_POP]
SAMP = ORDER[:N_SAMP]
HEAVY = ORDER[:N_HEAVY]
PIT_CHUNKS = [ORDER[i * N_PIT:(i + 1) * N_PIT] for i in range(N_PIT_CHUNKS)]

# The eval helpers index `val_ds[i]` positionally from 0, so a tier is materialised as
# its own dataset over the selected jets rather than passed as an index list.
jets_samp = [jets[i] for i in SAMP]
ds_samp = MatchedLundDataset(jets_samp, geom, AUX)
jets_heavy = [jets[i] for i in HEAVY]
ds_heavy = MatchedLundDataset(jets_heavy, geom, AUX)

print(f"tiers: POP {len(POP)}   PIT {N_PIT_CHUNKS} x {N_PIT}   "
      f"SAMP {len(SAMP)}   HEAVY {len(HEAVY)}  (nested prefixes of one shuffled order)")


In [ ]:
# THE one sampling pass. v0 drew K_DRAWS posterior samples for this same tier FOUR
# separate times — §3 occupancy, §4 `run_calibration`, §5 `run_closure`, §5 `collect()` —
# and shared none of them, with §8 paying for a fifth, smaller set. Here they are drawn
# once and handed to every section: ~24 min cheaper, and the sections become exactly
# PAIRED, so a difference between two of them can no longer be sampling noise.
#
# HEAVY is a prefix of SAMP (the cell above), so `DRAWS[:len(ds_heavy)]` IS the HEAVY
# tier's draws — the continuous rows need no second pass either.
seed_everything(SEED)     # `sample_batch` rides the GLOBAL torch RNG (no generator arg)
_t0 = time.perf_counter()
DRAWS = []
with torch.inference_mode():
    for k in range(len(ds_samp)):
        item = ds_samp[k]
        DRAWS.append(model.sample_batch(item["xf"].unsqueeze(0).to(device),
                                        torch.tensor([item["nx"]], device=device),
                                        K_DRAWS))
_dt = time.perf_counter() - _t0
assert len(DRAWS) == len(ds_samp) == len(jets_samp)
print(f"{len(DRAWS)} jets x {K_DRAWS} draws in {_dt / 60:.2f} min "
      f"({1e3 * _dt / max(len(DRAWS), 1):.1f} ms/jet)")
print(f"  mean drawn multiplicity {np.mean([len(c) for d in DRAWS for c in d]):.3f}"
      f"   empty draws {np.mean([len(c) == 0 for d in DRAWS for c in d]):.1%}")
print("  every section below reads THESE draws — nothing samples the SAMP tier again.")


In [ ]:
# The CLI counterpart, so this run is reproducible without the notebook.
print(f"""
h2p-rsd-junipr eval {rel(CKPT)} \\
    data=rntuple data.path={ROOT_PATH} \\
    experiment.pit_coords=true experiment.stratify_regions=true \\
    experiment.tarp=true experiment.closure_continuous=true
""".strip())
print("\n(naming `data` lifts it over the snapshot and evaluates the WHOLE file rather "
      "than a 10% split; `geometry` and `encoder` are deliberately not liftable.)")


## 2. The population at 30 bins, and what the two files agree on

The two files differ only by seed, so any disagreement in the multiplicity marginal
or the aux columns is a **generation bug, not physics**. What the agreement buys is
the substitute for the blocked PYTHIA-vs-HERWIG systematic: a same-generator,
different-seed **statistical noise floor** against which any future generator spread
must be read.


In [ ]:
def pop_summary(js, label):
    st = multiplicity_stats(js)
    nx = np.array([len(j["x"][0]) for j in js])
    w = np.array([j["weight"] for j in js], dtype=float)
    n_eff = w.sum() ** 2 / np.square(w).sum() if w.size else float("nan")
    return {
        "label": label, "n_jets": st["n_jets"],
        "mean_ny": st["mean"], "max_ny": st["max"], "p99_ny": st["p99"],
        "p_empty_y": st["frac_empty"], "mean_nx": float(nx.mean()),
        "p_empty_x": float((nx == 0).mean()),
        "mean_nsec": float(np.mean([j["x_nsec"] for j in js])),
        "p_nsec_zero": float(np.mean([j["x_nsec"] == 0 for j in js])),
        "mean_jet_pt": float(np.mean([j["jet_pt"] for j in js])),
        # check 13: eval/ averages UNWEIGHTED over val_ds while the distribution-closure
        # notebook weights everything. If the weights are non-trivial, every eval/ number
        # is an unweighted average of a weighted sample and must be labelled so.
        "n_unique_weights": int(np.unique(w).size),
        "n_eff": float(n_eff), "n_eff_over_n": float(n_eff / max(w.size, 1)),
    }


POPSTAT = {"test": pop_summary([jets[i] for i in POP], "test"),
           "train": pop_summary(train_jets, "train")}
hdr = ["n_jets", "mean_ny", "p_empty_y", "mean_nx", "p_empty_x", "mean_nsec",
       "p_nsec_zero", "mean_jet_pt"]
print(f"{'':<8}" + "".join(f"{h:>13}" for h in hdr))
for k, s in POPSTAT.items():
    print(f"{k:<8}" + "".join(
        f"{s[h]:>13.4f}" if isinstance(s[h], float) else f"{s[h]:>13d}" for h in hdr))

d = {h: POPSTAT["test"][h] - POPSTAT["train"][h] for h in hdr[1:]}
drel = {h: (d[h] / POPSTAT["train"][h] if POPSTAT["train"][h] else float("nan")) for h in d}
pad = " " * (8 + 13)   # skip the label and the n_jets column, which has no delta
print(f"{'delta':<8}" + pad[8:] + "".join(f"{d[h]:>13.4f}" for h in hdr[1:]))
print(f"{'rel':<8}" + pad[8:] + "".join(f"{drel[h]:>13.2%}" for h in hdr[1:]))
METRICS["train_test_agreement"] = {"delta": d, "relative": drel}
print("\nThese deltas ARE the same-generator different-seed noise floor: a future "
      "generator systematic\n(PYTHIA vs HERWIG) is only a finding where it exceeds them. "
      "WP5 has no herwig_driver, so\nthat systematic stays the known gap, recorded not "
      "silently skipped.")


In [ ]:
# The asymmetric floor's whole point, measured: x_nsec is no longer near-binary.
nsec_test = np.array([j["x_nsec"] for j in [jets[i] for i in POP]])
print(f"x_nsec on the test file: mean {nsec_test.mean():.3f},  zero fraction "
      f"{(nsec_test == 0).mean():.1%},  max {nsec_test.max()}")
print("the symmetric reference file (docs/PLAN_Input.md) had mean 0.25 and zero "
      "fraction 82.6%,\nwhich is why its aux A/B measured noise: the five "
      "secondary-plane features were\nconstant-zero four times in five.")
print("\nTRAP (check 5): `x_mg` / `x_ptg` are REDEFINED by this card. Their documented "
      "job is to be\nthe complement of the recorded sequence; under an asymmetric floor "
      "they are the complement\nof a tree the model never sees. Legitimate conditioning, "
      "but NOT the same feature as on\nthe symmetric file — do not compare their learned "
      "effect across the two.")


In [ ]:
# check 13, stated where it bites: `run_closure`/`run_calibration` average unweighted.
w_test = np.array([jets[i]["weight"] for i in POP], dtype=float)
if POPSTAT["test"]["n_unique_weights"] == 1:
    print(f"per-jet weights are constant ({w_test[0]:g}) — unweighted averaging in "
          f"eval/ is exact here,\nand n_eff = n = {len(w_test)}.")
else:
    print(f"WARNING: {POPSTAT['test']['n_unique_weights']} distinct weights, "
          f"n_eff = {POPSTAT['test']['n_eff']:.0f} of {len(w_test)} "
          f"({POPSTAT['test']['n_eff_over_n']:.1%}).\n"
          f"eval/ averages UNWEIGHTED over val_ds, so every eval/ number below is an "
          f"unweighted average\nof a weighted sample and must be labelled so. "
          f"lund_distribution_closure_v2.ipynb weights everything.")

sup = check_multiplicity_support([jets[i] for i in POP], cfg, strict=False)
print(f"\nmultiplicity support: max N = {sup['max']}, model.max_emissions = "
      f"{sup['support']}, P(N > support) = {sup['tail_fraction']:.2e}")
METRICS["population"] = {**POPSTAT, "support_guard": {
    "max_N": sup["max"], "model_max_emissions": sup["support"],
    "tail_fraction": sup["tail_fraction"]}}


## 2b. Held-out NLL, and which of its terms survived the geometry change

It is tempting to declare a 30-bin NLL simply incomparable to the 10-bin `4.61`. That
is too strong. With `continuous_coords: true` the **total** is a density on the
(ln 1/ΔR, ln k_t) plane and *is* dimensionally commensurable across `n_bins`; what is
not is `split_ll` **alone**, which is a probability over cells and shifts by
`2·ln(30/10) = 2.197` nat per emission for free.

The real confound on the comparable total is that a finer grid is a **strictly richer
density class** — so a lower 30-bin total is evidence of better *resolution*, not of a
better conditional.


In [ ]:
@torch.inference_mode()
def nll_terms_over(indices, batch_size=POP_BATCH):
    """Batched per-term NLL over a tier. Returns per-jet arrays."""
    out = {k: [] for k in ("length_ll", "split_ll", "coord_ll", "n_emissions")}
    for s in range(0, len(indices), batch_size):
        chunk = [jets[i] for i in indices[s:s + batch_size]]
        # ONE dataset per chunk. With the constructor INSIDE the comprehension it is
        # rebuilt once per k -- B datasets of B jets, O(B^2), measured 7.5 ms/jet against
        # 0.036 and ~60 min over the ablation's five arms. Hoisting it changes no number
        # (the dataset is deterministic and no RNG is involved), only the wall-clock.
        ds_chunk = MatchedLundDataset(chunk, geom, AUX)
        b = collate([ds_chunk[k] for k in range(len(chunk))])
        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        t = model.nll_terms(b)
        for k in out:
            out[k].append(t[k].detach().cpu().numpy())
    return {k: np.concatenate(v) for k, v in out.items()}


T = nll_terms_over(POP)
n_em = T["n_emissions"].sum()
nll_total = float(-(T["length_ll"] + T["split_ll"] + T["coord_ll"]).mean())
assert cfg.model.cell_label_smoothing == 0.0, (
    f"cell_label_smoothing={cfg.model.cell_label_smoothing} != 0: the split term is a "
    f"smoothed surrogate, not a log-probability, and is not comparable to anything."
)

shift = 2 * math.log(geom.n_bins / 10.0)
rows = [
    ("total NLL/jet", nll_total, "per jet", True,
     "density on the plane: cell-prob x within-cell density"),
    ("length -ln q(N|x)", float(-T["length_ll"].mean()), "per jet", True,
     "a distribution over N; references no cell grid"),
    ("split -ln q(cell)", float(-T["split_ll"].sum() / n_em), "per emission", False,
     f"probability over {geom.n_cells} cells; shifts by 2*ln(30/10) = +{shift:.3f}"),
    ("coord -ln p(du,dv,lnz,psi)", float(-T["coord_ll"].sum() / n_em), "per emission", False,
     f"density over a cell {geom.cell_wu:.2f} wide; pays the split shift back"),
    ("split+coord", float(-(T["split_ll"] + T["coord_ll"]).sum() / n_em), "per emission",
     True, "the product IS a density on the plane"),
]
print(f"{'term':<28}{'value':>10}  {'unit':<14}{'10-bin-comparable?':<20}why")
for name, val, unit, comparable, why in rows:
    print(f"{name:<28}{val:>10.4f}  {unit:<14}{('YES' if comparable else 'NO'):<20}{why}")
print(f"\n{int(n_em)} emissions over {len(POP)} jets "
      f"({n_em / len(POP):.3f} per jet).")
print("Caveat that survives the table: a finer grid is a strictly RICHER density class,"
      "\nso a lower 30-bin total is evidence of better resolution, not of a better "
      "conditional.\nAny run with cell_label_smoothing > 0, or any ar_junipr_v1 "
      "(continuous_coords: false),\nis excluded from the comparable column entirely.")

METRICS["nll"] = {
    "n_jets": len(POP), "n_emissions": int(n_em),
    "total_per_jet": nll_total,
    "length_per_jet": float(-T["length_ll"].mean()),
    "split_per_emission": float(-T["split_ll"].sum() / n_em),
    "coord_per_emission": float(-T["coord_ll"].sum() / n_em),
    "split_plus_coord_per_emission": float(-(T["split_ll"] + T["coord_ll"]).sum() / n_em),
    "split_ll_shift_vs_10_bins": shift,
    "cell_label_smoothing": float(cfg.model.cell_label_smoothing),
    "comparable_across_n_bins": {"total_per_jet": True, "length_per_jet": True,
                                 "split_per_emission": False, "coord_per_emission": False,
                                 "split_plus_coord_per_emission": True},
}


## 2c. The aux ablation — the actual scientific question

The whole reason this file exists is that the secondary-plane features stop being
constant-zero. Measuring that means a held-out NLL comparison of `aux_features=[...]`
against `aux_features=[]` — **with a seed band, or it cannot conclude**. The previous
A/B failed precisely because its −0.029 nat delta *was* the 0.029 seed spread; a
single on/off pair here would reproduce that ambiguity exactly. So this is a
two-by-two: aux on/off × seed 0/1.

Two strata are reported separately, because both are controls rather than decoration:

* **`nx == 0`** — aux rides as *constant per-node columns of `xf`*, so a jet with an
  empty groomed hadron tree receives **no aux signal at all** (the broadcast has no
  rows). If those jets "gain" as much as the rest, the aggregate delta is seed noise
  whatever its sign. This is the control that exposed the last A/B.
* **`n_sec`** — the symmetric file's only surviving signal was the `n_sec = 2–3`
  stratum at −0.100 nat.

**What this measures, precisely.** The nine aux columns are of three kinds, and the A/B
moves all of them at once: five describe the secondary plane (`nsec`, `has_sec`,
`ln_kt_sec`, `ln_kt_sec_sum`, `sec_depth`) and are degenerate when `n_sec == 0`; two
(`ln_mg_pt`, `ln_ptg_pt`) are the groomed mass/momentum, which the asymmetric floor
*also* redefines; and two (`ln_pt`, `abs_eta`) are floor-independent jet kinematics
defined for every jet with a node. So a gain in the `n_sec == 0` stratum is not a
contradiction — four columns are still live there — but it does mean the verdict below
is about **the nine-column set**, not about the secondary-plane features alone.
Isolating those needs a third arm with `aux_features=[ln_pt, abs_eta]`.


In [ ]:
def arm_checkpoints(root):
    """{label: ckpt} for the ablation grid, plus the headline arm as aux/seed 0."""
    arms = {"aux_s0": CKPT}
    root = REPO / root
    for d in sorted(root.glob("*")) if root.exists() else []:
        ck = sorted(d.glob("*/best.ckpt"), key=lambda p: p.stat().st_mtime)
        if ck:
            arms[d.name] = ck[-1]
    return arms


@torch.inference_mode()
def per_jet_nll_of(ckpt, indices, batch_size=POP_BATCH):
    """Per-jet NLL of one checkpoint on a fixed jet list. Each arm builds its OWN
    dataset from its own `aux_feature_names`, so the aux-off arm gets a 5-wide `xf`
    and the aux-on arm a 14-wide one — the same jets, each fed as its model expects."""
    inf = load_for_inference(ckpt, map_location=device)
    c = OmegaConf.create(inf["config"])
    g = Geometry.from_config(c.geometry)
    m = build_model(c, g).to(device)
    m.load_state_dict(inf["model_state"])
    m.eval()
    aux = tuple(m.aux_feature_names)
    out = []
    for s in range(0, len(indices), batch_size):
        chunk = [jets[i] for i in indices[s:s + batch_size]]
        ds_chunk = MatchedLundDataset(chunk, g, aux)   # once per chunk -- see 2b
        b = collate([ds_chunk[k] for k in range(len(chunk))])
        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        out.append(m.per_jet_nll(b).detach().cpu().numpy())
    # `inf["epoch"]` is the epoch the BEST checkpoint was written at, not the schedule
    # length — `best.ckpt` is saved whenever val improves, so two arms that both ran 60
    # epochs report 49 and 45. The schedule is `trainer.max_epochs`, and confusing the
    # two makes the guard below fire on every honest grid.
    return np.concatenate(out), {"aux": list(aux), "best_epoch": inf["epoch"],
                                 "max_epochs": int(c.trainer.max_epochs),
                                 "best_val_nll": inf["best_val_nll"]}


ARMS = arm_checkpoints(ABLATION_ROOT)
print("ablation arms:")
for k, v in ARMS.items():
    print(f"  {k:<12} {rel(v)}")

# The WHOLE population, not a subsample. This is a batched forward pass per arm, so the
# only cost of capping it is noise — and the noise lands on the SEED BAND, which is a max
# of two differences of two noisy means and is therefore the most fragile number here. A
# 40k subsample was enough to move the band from 0.028 to 0.058 and flip the verdict while
# the delta itself moved by 0.003.
ABL_IDX = POP                                         # paired across every arm
NLLS, ARMINFO = {}, {}
for label, ck in ARMS.items():
    NLLS[label], ARMINFO[label] = per_jet_nll_of(ck, ABL_IDX)
    a = ARMINFO[label]
    print(f"  {label:<12} held-out NLL/jet = {NLLS[label].mean():.4f}   "
          f"aux={'on ' if a['aux'] else 'OFF'}  "
          f"({a['max_epochs']} epochs scheduled, best at {a['best_epoch']})")


In [ ]:
AUX_ON = sorted(k for k in NLLS if ARMINFO[k]["aux"])
AUX_OFF = sorted(k for k in NLLS if not ARMINFO[k]["aux"])

# An arm trained for fewer epochs is a different experiment, and the difference lands
# in the same column as the aux effect. Refuse to read the grid as an ablation unless
# every arm ran the same schedule.
_epochs = {k: ARMINFO[k]["max_epochs"] for k in NLLS}
SAME_SCHEDULE = len(set(_epochs.values())) == 1
if not SAME_SCHEDULE:
    print(f"WARNING: arms trained on DIFFERENT schedules {_epochs}. Every delta below "
          f"mixes\nthe aux effect with a training-length effect and is NOT an "
          f"ablation. Re-run the short arms.")
ABL = {"arms": {k: {"nll": float(v.mean()), **ARMINFO[k],
                    "checkpoint": str(rel(ARMS[k]))} for k, v in NLLS.items()},
       "n_jets": len(ABL_IDX), "aux_on": AUX_ON, "aux_off": AUX_OFF,
       "same_schedule": SAME_SCHEDULE, "epochs": _epochs}

if len(AUX_ON) >= 2 or len(AUX_OFF) >= 2:
    # The seed band: the spread WITHIN an arm type, which is the only scale against
    # which the between-arm delta means anything.
    bands = {t: (max(NLLS[k].mean() for k in ks) - min(NLLS[k].mean() for k in ks))
             for t, ks in (("aux_on", AUX_ON), ("aux_off", AUX_OFF)) if len(ks) >= 2}
    band = max(bands.values())
    ABL["seed_band"] = {"per_arm": bands, "band": float(band)}
    print(f"seed band (max within-arm spread over seeds): {band:.4f} nat/jet   {bands}")
else:
    band = None
    print("FEWER THAN TWO SEEDS OF EITHER ARM — no seed band, so no delta below is "
          "rankable.\nAn underpowered A/B is worse than none, because it will be quoted. "
          "Reporting the\narm on its own is the honest fallback.")

if AUX_ON and AUX_OFF:
    on = np.mean([NLLS[k] for k in AUX_ON], axis=0)
    off = np.mean([NLLS[k] for k in AUX_OFF], axis=0)
    delta = float((on - off).mean())          # negative == aux helps
    # paired over the SAME jets, so the per-jet standard error is the right one
    sem = float((on - off).std(ddof=1) / math.sqrt(len(ABL_IDX)))
    ABL["delta_nat_per_jet"] = delta
    ABL["delta_sem"] = sem
    print(f"\naux ON - aux OFF = {delta:+.4f} nat/jet   (paired SEM {sem:.4f}, "
          f"{len(ABL_IDX)} jets)")
    if band is not None:
        verdict = ("AUX HELPS beyond the seed band" if delta < -band else
                   "AUX HURTS beyond the seed band" if delta > band else
                   "INSIDE the seed band — undecided, exactly the previous A/B's failure")
        print(f"seed band {band:.4f}  =>  {verdict}")
        ABL["verdict"] = verdict
        # How much to trust that verdict. The band is `max` of |s0 - s1| over two arms,
        # i.e. a maximum of two differences of two noisy means — upward-biased, and with
        # only two seeds per arm it has no meaningful error bar of its own. When |delta|
        # and the band are the same order, the honest reading is "suggestive", and the
        # fix is more seeds, not a different cut.
        ratio = abs(delta) / max(band, 1e-12)
        ABL["delta_over_band"] = ratio
        ABL["band_is_two_seeds"] = True
        # A range from two samples carries ~60% relative uncertainty, so the band is a
        # soft threshold, not a sharp one. Below 3x, say so.
        if ratio < 3.0:
            print(f"  CAUTION: |delta| / band = {ratio:.2f}. The band is `max` over two "
                  f"arms of a TWO-SEED range,\n  which has roughly 60% relative "
                  f"uncertainty of its own — a third seed could move it by\n  half its "
                  f"size in either direction. The paired SEM ({sem:.4f}) bounds the "
                  f"DELTA well;\n  nothing here bounds the BAND. Treat the verdict as "
                  f"suggestive and read the `nx == 0`\n  control below, which does not "
                  f"depend on the band at all.")
        else:
            print(f"  |delta| / band = {ratio:.2f} — the delta clears the band by a "
                  f"comfortable margin.")

    # --- the strata that decide whether the aggregate means anything -----------
    nx_arr = np.array([len(jets[i]["x"][0]) for i in ABL_IDX])
    nsec_arr = np.array([jets[i]["x_nsec"] for i in ABL_IDX])
    print(f"\n{'stratum':<18}{'jets':>8}{'aux on':>10}{'aux off':>10}{'delta':>10}"
          f"{'vs band':>12}")
    strata = [("nx == 0 (CONTROL)", nx_arr == 0), ("nx > 0", nx_arr > 0),
              ("n_sec == 0", nsec_arr == 0), ("n_sec 1", nsec_arr == 1),
              ("n_sec 2-3", (nsec_arr >= 2) & (nsec_arr <= 3)),
              ("n_sec >= 4", nsec_arr >= 4)]
    ABL["strata"] = {}
    for name, sel in strata:
        if not sel.any():
            continue
        dd = float((on[sel] - off[sel]).mean())
        tag = "-" if band is None else ("beyond" if abs(dd) > band else "inside")
        print(f"{name:<18}{int(sel.sum()):>8}{on[sel].mean():>10.4f}"
              f"{off[sel].mean():>10.4f}{dd:>+10.4f}{tag:>12}")
        ABL["strata"][name] = {"n": int(sel.sum()), "aux_on": float(on[sel].mean()),
                               "aux_off": float(off[sel].mean()), "delta": dd}
    print("\nThe `nx == 0` row is the control, and it does NOT depend on the seed band. "
          "Those jets\ncarry NO aux signal (aux is a constant per-node column and there "
          "are no nodes), so a\ngain there is arithmetically impossible to attribute to "
          "aux. If it moves like the rest,\nthe aggregate is seed noise whatever the "
          "band says; if it stays flat or moves the OTHER\nway while `nx > 0` gains, "
          "that is positive evidence no amount of seed spread explains.")
else:
    print("\nonly one arm type present — no ablation possible.")

METRICS["aux_ablation"] = ABL


## 3. Split-head occupancy — can 900 cells be filled, and can the head reach them?

Two separate questions that a single "900 cells" does not answer.

**Occupancy.** The Lund density is peaked, so the ~940 emissions/cell the plan sizes
is an *average* that overstates the sparse corners. What matters is how many cells the
truth actually occupies and how many the posterior ever emits: emitting 300 where
truth occupies 420 is **under-covering the support**.

**Rank (check 10).** `split_head` ends in `Linear(64, 900)`, so the logit map has
**rank ≤ 64 over a 900-dimensional output** — a constraint with no analogue at 100
cells. If the effective rank is saturated *and* the model under-covers, the answer is
a wider `dec_dim` or another `split_head` layer, not more data.


In [ ]:
truth_cells = np.concatenate([geom.seq_cells(jets[i]["y"][0], jets[i]["y"][1])
                              for i in POP]) if len(POP) else np.array([], dtype=int)
occ_truth = np.bincount(truth_cells, minlength=geom.n_cells)

# the shared draws from §1 — this used to be a sampling pass of its own
post_cells = [int(c) for draws in DRAWS for d in draws for c in d]
occ_post = np.bincount(np.array(post_cells, dtype=int), minlength=geom.n_cells)

W = model.split_head[-1].weight.detach().cpu().double().numpy()
sv = np.linalg.svd(W, compute_uv=False)
# effective rank = exp(entropy of the normalised singular spectrum) — the standard
# continuous relaxation of rank, and the honest way to say "64 available, N used".
p_sv = sv / sv.sum()
eff_rank = float(np.exp(-(p_sv * np.log(p_sv + 1e-300)).sum()))
sv99 = int(np.searchsorted(np.cumsum(sv ** 2) / np.sum(sv ** 2), 0.99) + 1)

print(f"split_head final layer: {W.shape[1]} -> {W.shape[0]}   "
      f"hard rank bound = {min(W.shape)}")
print(f"  effective rank (spectral entropy) = {eff_rank:.1f} of {min(W.shape)}"
      f"   ({eff_rank / min(W.shape):.0%} of the bound)")
print(f"  singular values holding 99% of the energy: {sv99}")
print(f"\ncell occupancy over {geom.n_cells} cells:")
print(f"  truth ({len(truth_cells)} emissions, {len(POP)} jets): "
      f"{int((occ_truth > 0).sum())} cells occupied "
      f"({(occ_truth > 0).mean():.1%}), median count {int(np.median(occ_truth[occ_truth > 0]))}")
print(f"  posterior ({len(post_cells)} emissions, {len(SAMP)} jets x {K_DRAWS} draws): "
      f"{int((occ_post > 0).sum())} cells emitted "
      f"({(occ_post > 0).mean():.1%})")
covered = int(((occ_truth > 0) & (occ_post > 0)).sum())
missed = int(((occ_truth > 0) & (occ_post == 0)).sum())
print(f"  of the {int((occ_truth > 0).sum())} truth-occupied cells the posterior reaches "
      f"{covered} and MISSES {missed}")
print(f"  cells the posterior emits that truth never occupies: "
      f"{int(((occ_post > 0) & (occ_truth == 0)).sum())}")
if missed > 0.1 * max((occ_truth > 0).sum(), 1):
    print("\n  => the posterior UNDER-COVERS the support. With the rank bound above, "
          "that argues\n     for widening dec_dim or adding a split_head layer, not for "
          "more data.")

METRICS["occupancy"] = {
    "n_cells": geom.n_cells,
    "split_head_shape": [int(W.shape[1]), int(W.shape[0])],
    "rank_bound": int(min(W.shape)), "effective_rank": eff_rank,
    "sv_99pct": sv99,
    "truth_emissions": int(len(truth_cells)),
    "truth_cells_occupied": int((occ_truth > 0).sum()),
    "posterior_emissions": int(len(post_cells)),
    "posterior_cells_emitted": int((occ_post > 0).sum()),
    "truth_cells_covered_by_posterior": covered,
    "truth_cells_missed_by_posterior": missed,
    "posterior_only_cells": int(((occ_post > 0) & (occ_truth == 0)).sum()),
}


## 4. Calibration — SBC, per-coordinate PIT, region strata, TARP

Two things this section does that the plan calls out as missing everywhere else:
every coverage number is quoted with a **Wilson interval and its count**, and every
uniformity statistic against its **null reference** (`sbc_chi2` vs the χ²(9) 95%
point, `tarp_max_dev` vs `≈1.36/√n`). A bare `0.66 vs 0.68` on ~300 jets is a coin
flip, and `PLAN_ProductionAssessment.md` §9.4's "no Lund quadrant fails the band"
cannot be asserted without them.

`coordinate_pits` **collates every jet into one padded batch**, so it is run on four
disjoint 5 000-jet chunks rather than the population: the chunks bound the memory and
their spread is a free MC-error estimate on the KS statistics.


In [ ]:
# `draws_by_jet` is what makes this free: SBC / PIT / coverage are numpy over draws
# that already exist (§1), so nothing here samples. The seed is kept so the cell is
# still reproducible on its own if DRAWS is rebuilt.
seed_everything(SEED)
CAL = run_calibration(
    model, ds_samp, geom, device, K=K_DRAWS, n_jets=len(ds_samp),
    pit_coords=False,              # done per chunk below instead
    stratify_regions=True, tarp=False, seed=SEED, min_region_n=30,
    draws_by_jet=DRAWS,
)


In [ ]:
# check 12, restated where it is decided: with the interval, is the quadrant claim real?
print("\nper-quadrant verdict (the claim PLAN_ProductionAssessment.md §9.4 wants to make):")
scored = [k for k, e in CAL.get("by_region", {}).items() if e["scored"]]
failing = [k for k in scored if not CAL["by_region"][k]["coverage_68_consistent"]]
print(f"  {len(scored)} of {len(CAL.get('by_region', {}))} quadrants have >= "
      f"{CAL['region_min_n']} jets and are scored: {scored}")
print(f"  quadrants whose 95% Wilson interval EXCLUDES 0.68: "
      f"{failing if failing else 'none'}")
print("  a quadrant below the floor is reported but NOT scored — its interval is wider "
      "than the\n  effect anyone would claim from it.")
# `HadronJet:R` is a card setting and is NOT stored in the RNTuple provenance, so it
# cannot be read off the file the way z_cut / beta / kt_floor can. Stated, not assumed.
JET_R = 0.4
u_min = math.log(1 / JET_R)
print(f"\n  note: u = ln(1/DeltaR) >= ln(1/R) = {u_min:.2f} at R = {JET_R} (from the "
      f"card, not the file), while the "
      f"quadrant split is at\n  u = {CAL['region_split']['u']:.1f} — so the low-u strip "
      f"of the `wide_*` quadrants is empty by\n  kinematics, not by the model. A "
      f"near-empty quadrant here is a geometry fact.")


In [ ]:
# Per-coordinate PIT on four disjoint chunks. The spread across chunks is the MC error.
PITS = []
for ci, idx in enumerate(PIT_CHUNKS):
    chunk = [jets[i] for i in idx]
    p = coordinate_pits(model, MatchedLundDataset(chunk, geom, AUX), geom, device,
                        n_jets=len(chunk), n_bins=10, stratify_regions=True,
                        verbose=(ci == 0))
    if p is None:
        print("this family has no exact coordinate density — per-coordinate PIT skipped.")
        break
    PITS.append(p)

if PITS:
    names = PITS[0]["names"]
    print(f"\nKS per coordinate across {len(PITS)} disjoint chunks of "
          f"{PIT_CHUNKS[0].size} jets  (mean +- spread):")
    print(f"{'coord':>6}{'KS mean':>10}{'KS spread':>12}{'mean(PIT)':>12}"
          f"{'95% crit':>10}   verdict")
    PITSUM = {}
    for nm in names:
        ks = np.array([p["coords"][nm]["ks"] for p in PITS])
        mu = np.array([p["coords"][nm]["mean"] for p in PITS])
        n_em = np.mean([p["coords"][nm]["n"] for p in PITS])
        c = 1.36 / math.sqrt(max(n_em, 1))
        v = "calibrated" if ks.mean() < c else "MISCALIBRATED"
        print(f"{nm:>6}{ks.mean():>10.4f}{ks.std(ddof=1) if len(ks) > 1 else 0:>12.4f}"
              f"{mu.mean():>12.4f}{c:>10.4f}   {v}")
        PITSUM[nm] = {"ks_mean": float(ks.mean()),
                      "ks_spread": float(ks.std(ddof=1)) if len(ks) > 1 else 0.0,
                      "pit_mean": float(mu.mean()), "n_emissions": float(n_em),
                      "ks_crit95": float(c), "calibrated": bool(ks.mean() < c)}
    print("\n`du`/`dv` are the within-cell offsets, whose support narrowed 3x with "
          "n_bins while\n`sigma_floor` was lowered 0.01 -> 0.005 to match. THIS is the "
          "measurement that says\nwhether that was necessary: a floor that binds shows "
          "up as a U-shaped (over-confident) PIT.")

    # the exposure-bias signature: is the LATE emission calibrated, or only the first?
    print(f"\nKS by emission index (chunk 0), the exposure-bias signature:")
    bi = PITS[0]["coords"][names[0]]["by_emission_index"]
    print(f"{'t':>4}" + "".join(f"{nm:>10}" for nm in names) + f"{'n':>9}")
    for t in sorted(bi, key=int):
        row = [PITS[0]["coords"][nm]["by_emission_index"].get(t, {}) for nm in names]
        print(f"{t:>4}" + "".join(f"{r.get('ks', float('nan')):>10.4f}" for r in row)
              + f"{row[0].get('n', 0):>9}")
    print("a KS that GROWS with t is the model losing calibration on its own generated "
          "prefix.")
else:
    PITSUM = {}

METRICS["calibration"] = {**{k: v for k, v in CAL.items() if k != "pit_coords"},
                          "pit_chunks": {"n_chunks": len(PITS), "per_chunk_jets": N_PIT,
                                         "coords": PITSUM}}


In [ ]:
# TARP: a JOINT test over the whole tree in the physics metric, on the HEAVY tier.
seed_everything(SEED)
from h2p_rsd_junipr.eval.calibration import run_tarp  # noqa: E402
from h2p_rsd_junipr.inference.mbr import mbr_kwargs_from_decode  # noqa: E402

MBRKW = mbr_kwargs_from_decode({**DECODE, "mbr_backend": MBR_BACKEND})
TARP = run_tarp(model, ds_heavy, geom, device, K=K_DRAWS, n_jets=len(ds_heavy),
                n_refs=100, reference="pooled", mbr_kwargs=MBRKW, seed=SEED)
METRICS["calibration"]["tarp"] = TARP


## 5. Closure and the leading emission

**`run_closure`'s own population is truth-selected** (check 9): `eval/closure.py` does
`if ly is None or not lead: continue`, so every `dlund_*` number is conditioned on the
truth having ≥1 emission. That is `p(leading | n_y > 0)`, not `p(leading)`, and the
kept fraction is reported below rather than left implicit. The empty-tree keys are
computed *before* that `continue` — §6 validates that independently, because a
`p_empty_true` of exactly `0.0` is the tell that the ordering regressed.


In [ ]:
seed_everything(SEED)
CLO = run_closure(model, ds_samp, jets_samp, geom, device, K=K_DRAWS,
                  n_closure=len(ds_samp), decode={**DECODE, "mbr_backend": MBR_BACKEND},
                  continuous=False, draws_by_jet=DRAWS)
print(f"\nkept fraction: {CLO['n_kept_leading']}/{CLO['n_jets_scored']} = "
      f"{CLO['n_kept_leading'] / CLO['n_jets_scored']:.1%} — every dlund_* number above "
      f"is\np(leading | n_y > 0), NOT p(leading).")


In [ ]:
# The continuous row, on the HEAVY tier: at 30 bins a cell is 0.2 while the distances
# are ~0.6, so the cell metric is no longer quantisation-limited — but the off-grid
# comparison is what the acceptance criterion is stated on.
# HEAVY is a prefix of SAMP, so its draws are the first len(ds_heavy) of §1's — the
# cell and continuous rows are computed on the SAME draws, not two independent sets.
# The coordinates are drawn one batched call per jet (`sample_coordinates_many`), which
# is the 36 min -> 0.3 min line of the budget table.
seed_everything(SEED)
CLO_C = run_closure(model, ds_heavy, jets_heavy, geom, device, K=K_DRAWS,
                    n_closure=len(ds_heavy),
                    decode={**DECODE, "mbr_backend": MBR_BACKEND}, continuous=True,
                    draws_by_jet=DRAWS[:len(ds_heavy)])


In [ ]:
# check 14: the decode-free comparator beside every point estimate, so a reviewer can
# separate a MODEL failure from a DECODE ceiling.
print("\ndecode-free comparator (posterior predictive) beside the point estimates:")
print(f"{'quantity':<34}{'identity(x)':>13}{'MAP/mode':>11}{'medoid':>10}"
      f"{'posterior':>12}")
print(f"{'leading-emission Lund distance':<34}{CLO['dlund_identity']:>13.4f}"
      f"{CLO['dlund_posterior_mode']:>11.4f}{CLO['dlund_posterior_medoid']:>10.4f}"
      f"{'--':>12}")
print(f"{'multiplicity signed bias':<34}{CLO['mult_bias_identity']:>13.4f}"
      f"{'--':>11}{'--':>10}{CLO['mult_bias_posterior']:>12.4f}")
print(f"{'P(n = 0)':<34}{'--':>13}{CLO['p_empty_pred']:>11.4f}{'--':>10}"
      f"{'see 6':>12}")
print(f"{'mean multiplicity':<34}{CLO['mean_mult_hadron']:>13.4f}{'--':>11}{'--':>10}"
      f"{CLO['mean_mult_posterior']:>12.4f}   (truth {CLO['mean_mult_true']:.4f})")


In [ ]:
# The estimator comparison at cell level and off-grid, via the SAME `collect()` the
# leading-estimator study uses. Imported by path rather than shelled out: its `main()`
# builds a LundDataModule from the CHECKPOINT's cfg.data.path and would silently score
# the training file's val split. `collect()` itself is sample-agnostic.
spec = importlib.util.spec_from_file_location(
    "leading_estimators", REPO / "scripts" / "leading_estimators.py")
le = importlib.util.module_from_spec(spec)
spec.loader.exec_module(le)

seed_everything(SEED)
R, C = le.collect(model, ds_samp, jets_samp, geom, device,
                  n_jets=len(ds_samp), K=K_DRAWS, n_cont=len(ds_heavy),
                  draws_by_jet=DRAWS)     # §1's draws again: paired with §3/§4/§5 above
base = float(np.nanmean(R[:, 0]))
print(f"\n=== cell level, {len(R)} jets with a truth leading emission "
      f"(cell {geom.cell_wu:.2f} wide) ===")
print(f"{'estimator':<26}{'mean d':>9}{'median d':>10}{'ratio':>8}{'exact':>8}")
LEAD = {"n_jets_cell": int(len(R)), "n_jets_cont": int(len(C))}
for j, nm in enumerate(["identity(x)", "mode", "medoid (loss-matched)", "oracle"]):
    m = float(np.nanmean(R[:, j]))
    ex = [None, float(R[:, 5].mean()), float(R[:, 6].mean()), None][j]
    print(f"{nm:<26}{m:>9.4f}{float(np.nanmedian(R[:, j])):>10.4f}{m / base:>8.3f}"
          f"{('--' if ex is None else f'{ex:.1%}'):>8}")
    LEAD[f"cell_{nm.split()[0]}"] = {"mean": m, "ratio": m / base, "exact": ex}

if len(C):
    cbase = float(np.nanmean(C[:, 0]))
    print(f"\n=== off the grid, {len(C)} jets ===")
    print(f"{'estimator':<26}{'mean d':>9}{'ratio':>8}")
    for j, nm in enumerate(["identity(x)", "cell centre of mode", "geometric median",
                            "posterior mean", "oracle"]):
        m = float(np.nanmean(C[:, j]))
        print(f"{nm:<26}{m:>9.4f}{m / cbase:>8.3f}")
        LEAD[f"cont_{nm.split()[0]}"] = {"mean": m, "ratio": m / cbase}

print("\nthe oracle is min over draws, so it measures whether the SUPPORT covers the "
      "truth —\nit is not an achievable point-estimate ceiling.")


In [ ]:
# Stratified by truth leading ln kt. A ratio > 1 in the SOFT bin is the signature to
# chase: a calibrated fully-conditional posterior beats any function of x, so losing to
# identity there means the model is under-conditioning in the soft/wide-angle corner.
qs = np.quantile(R[:, 4], [0.0, 1 / 3, 2 / 3, 1.0])
print(f"\n{'ln kt bin':<20}{'jets':>6}{'identity':>10}{'mode':>10}{'medoid':>10}"
      f"{'medoid/id':>11}")
LEAD["by_leading_lnkt"] = []
for lo, hi in zip(qs[:-1], qs[1:]):
    m = (R[:, 4] >= lo) & (R[:, 4] <= hi)
    idm, mdm = float(np.nanmean(R[m, 0])), float(np.nanmean(R[m, 2]))
    print(f"[{lo:5.2f}, {hi:5.2f}]{'':<7}{int(m.sum()):>6}{idm:>10.4f}"
          f"{float(np.nanmean(R[m, 1])):>10.4f}{mdm:>10.4f}{mdm / idm:>11.3f}")
    LEAD["by_leading_lnkt"].append({"lo": float(lo), "hi": float(hi), "n": int(m.sum()),
                                    "identity": idm, "medoid": mdm, "ratio": mdm / idm})

METRICS["closure"] = {"cell_tier": CLO, "continuous_tier": CLO_C, "leading": LEAD,
                      "kept_fraction": CLO["n_kept_leading"] / CLO["n_jets_scored"],
                      "conditioning": "p(leading | n_y > 0)"}


## 6. The empty parton tree

The parton target genuinely *is* the empty tree for roughly one jet in six, and no
point estimator under the default decode can say so. Four things this section
separates, which the single number `p_empty_pred` conflates:

1. **Is the ranking good?** `q(0|x)` AUC — can the model tell the two classes apart at
   all.
2. **Is the scale right?** A **reliability diagram** and a **Brier score with its
   reliability/resolution decomposition**. SBC/PIT *cannot* catch a scale error here,
   because SBC ranks against the sampler's own draws.
3. **Does recalibration fix it?** `(T, tilt)` from `fit_length_recalibration`. A
   scalar temperature alone **cannot**: it is symmetric about the mode, so it pulls
   `q(0|x)` *down* toward `1/(max_emissions+1)`, while the measured error is a
   monotone ramp across `n`. Both the uncalibrated and recalibrated rows are shown, or
   the recalibration hides the defect it corrects.
4. **Can the gate reproduce the rate?** `tau` from `empty_threshold_for_rate`.

**Both fits are made on the TRAINING file's val split and applied frozen here.**
`empty_threshold_for_rate` is a quantile of `q(0|x)` and reproduces its fitted rate
*by construction*, so fitting and reporting on the same jets would measure the
quantile function rather than the model. This is exactly what an independent test file
is for, and it is the one place the empty-tree plan leaves the protocol implicit.


In [ ]:
from h2p_rsd_junipr.data.datamodule import LundDataModule  # noqa: E402

# The training run's OWN val split, reproduced from its config — same seed, same
# event-level split. These jets are held out from training but are NOT the test file.
N_FIT = 20000     # enough for a stable quantile and a 2-parameter fit; the cost is
#                   one forward pass per jet, and the val split is ~50k

# `cfg.data.path` is stored REPO-relative, and a notebook kernel runs in `notebooks/`.
# Resolve it, or `load_rntuple` misses the file — and it does not raise when it does:
# it prints a note and hands back None, at which point the datamodule substitutes
# SYNTHETIC jets. This section would then fit tau and (T, tilt) on synthetic data and
# label the result "the training file's val split". Hence the assert below.
cfg_fit = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))
cfg_fit.data.path = str(REPO / str(cfg.data.path))
dm_fit = LundDataModule(cfg_fit, geom).setup()
assert dm_fit.jets and dm_fit.jets[0].get("generator") != "synthetic", (
    f"LundDataModule fell back to SYNTHETIC jets — {cfg_fit.data.path} was not readable. "
    f"Every number in this section would describe synthetic data while claiming to "
    f"describe the training file's val split."
)
assert len(dm_fit.jets) == len(train_jets), (
    f"the datamodule read {len(dm_fit.jets)} jets from the training file but section 1 "
    f"read {len(train_jets)} — these are not the same sample."
)
# Subsampled with the notebook seed, NOT a prefix: `_split_by_event` groups jets by
# event, so the first N of val_jets is the first N events, not N random jets.
_fit_order = np.random.default_rng(SEED).permutation(len(dm_fit.val_jets))[:N_FIT]
fit_jets = [dm_fit.val_jets[int(i)] for i in _fit_order]
ds_fit = MatchedLundDataset(fit_jets, geom, AUX)
print(f"recalibration/threshold fit set: {len(fit_jets)} of {len(dm_fit.val_jets)} jets "
      f"from the TRAINING\nfile's val split, disjoint from the {len(jets)} test jets "
      f"below.")
print(f"  datamodule fingerprint here: {dm_fit.fingerprint}  (the training run recorded a "
      f"different one\n  only because `_fingerprint` hashes the PATH STRING and this cell "
      f"absolutised it; the split\n  itself depends on data.seed and the event ids, not "
      f"the path, so these are the same jets)")


N_PMF_CHECK = 500   # jets on which the batched helper is checked against batch-1


@torch.inference_mode()
def length_pmfs(dset, n_max=None, batch_size=POP_BATCH):
    """Uncalibrated P(n|x) per jet, plus the true multiplicity. `length_temperature` /
    `length_tilt` are zeroed for the duration so the fit sees the RAW head — otherwise
    a checkpoint carrying a previous fit would be recalibrated twice.

    BATCHED (v1). For the multiplicity-head model `length_pmf` is
    `softmax(n_head(encode(x)))` — a pure batched op that v0 called 40 000 times at
    batch 1 for 13 min of wall-clock. Chunked at POP_BATCH through the same `collate`
    §2b already uses, it is ~3 s. No RNG is involved, so this is bit-identical rather
    than close, and the cell below asserts exactly that on N_PMF_CHECK jets rather than
    trusting it. `recalibrated_n_logits` is applied (not bypassed) so the helper stays
    correct if the zeroing above is ever removed.

    A family with no explicit head has no such identity — its length belief IS the
    sampler histogram — so that path keeps the per-jet loop."""
    t0, b0 = model.length_temperature, model.length_tilt
    model.length_temperature, model.length_tilt = 1.0, 0.0
    try:
        n = len(dset) if n_max is None else min(n_max, len(dset))
        ns = np.array([int(dset[k]["ny"]) for k in range(n)], dtype=int)
        if not hasattr(model, "n_head"):
            pmfs = [model.length_pmf(dset[k]["xf"].unsqueeze(0).to(device),
                                     torch.tensor([dset[k]["nx"]], device=device))
                    for k in range(n)]
            return pmfs, ns
        pmfs = []
        for s in range(0, n, batch_size):
            b = collate([dset[k] for k in range(s, min(s + batch_size, n))])
            b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
            e = model.encode(b["xf"], b["nx"])
            p = torch.softmax(model.recalibrated_n_logits(model.n_head(e)), dim=-1)
            pmfs.extend(p.detach().cpu().numpy())
        return pmfs, ns
    finally:
        model.length_temperature, model.length_tilt = t0, b0


_t0 = time.perf_counter()
pmf_fit, n_fit = length_pmfs(ds_fit)
ds_test_pmf = MatchedLundDataset([jets[i] for i in POP[:20000]], geom, AUX)
pmf_test, n_test = length_pmfs(ds_test_pmf)
print(f"fit on {len(pmf_fit)} jets, report on {len(pmf_test)} test jets "
      f"({time.perf_counter() - _t0:.1f} s batched; v0's batch-1 loop was ~13 min).")


In [ ]:
# The batched helper above must reproduce the batch-1 path EXACTLY — no RNG is involved,
# so "within noise" would be the wrong claim and a mismatch means the collated padding is
# reaching the encoder. That would also silently corrupt §2b's NLL, which is batched the
# same way, so this is the one check that covers both.
_t0 = time.perf_counter()
with torch.inference_mode():
    _t, _b = model.length_temperature, model.length_tilt
    model.length_temperature, model.length_tilt = 1.0, 0.0
    try:
        _ref = [model.length_pmf(ds_test_pmf[k]["xf"].unsqueeze(0).to(device),
                                 torch.tensor([ds_test_pmf[k]["nx"]], device=device))
                for k in range(min(N_PMF_CHECK, len(ds_test_pmf)))]
    finally:
        model.length_temperature, model.length_tilt = _t, _b
_bad = [k for k, p in enumerate(_ref) if not np.array_equal(p, pmf_test[k])]
assert not _bad, (
    f"batched length_pmf differs from the batch-1 path on {len(_bad)} of {len(_ref)} jets "
    f"(first: {_bad[:3]}). The batched encoder is seeing the collate padding."
)
print(f"batched length_pmf is BIT-IDENTICAL to the batch-1 path on {len(_ref)} jets "
      f"({time.perf_counter() - _t0:.1f} s to check, which is the cost of the path it "
      f"replaces).")


In [ ]:
# --- 1. ranking: can the model tell the two classes apart? -------------------
def auc(scores, labels):
    """Mann-Whitney AUC with mid-ranks for ties (no scipy)."""
    s, y = np.asarray(scores, float), np.asarray(labels, bool)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return float("nan")
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s), float)
    sr = s[order]
    i = 0
    while i < len(sr):
        j = i
        while j + 1 < len(sr) and sr[j + 1] == sr[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return float((ranks[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


q0_test = np.array([p[0] for p in pmf_test])
is0_test = n_test == 0
AUC = auc(q0_test, is0_test)
print(f"q(0|x) AUC = {AUC:.3f}   (0.5 == no information; the ranking is what tau uses)")

# --- 2. scale: reliability + Brier with its decomposition --------------------
def brier_decomposition(p, y, n_bins=10):
    """Brier = reliability - resolution + uncertainty (Murphy 1973). `reliability` is
    the term a miscalibrated SCALE inflates while the ranking (and hence the AUC) stays
    untouched — the one number that separates 'the ranking is good' from 'the scale is
    wrong'.

    Binned by QUANTILE of `p`, not by fixed width. The decomposition is valid for any
    partition, and `q(0|x)` is under-confident enough that fixed 0.1-wide bins put every
    jet in the first two — a diagram with two points, which says nothing about where the
    ranking is good. Equal-count bins spend the resolution where the data is."""
    p, y = np.asarray(p, float), np.asarray(y, float)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    n_bins = max(len(edges) - 1, 1)
    base = y.mean()
    rel = res = 0.0
    curve = []
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        nk, pk, ok = int(m.sum()), float(p[m].mean()), float(y[m].mean())
        rel += nk * (pk - ok) ** 2
        res += nk * (ok - base) ** 2
        lo, hi = wilson_interval(y[m].sum(), nk)
        curve.append({"bin": b, "n": nk, "p_mean": pk, "observed": ok,
                      "ci": [lo, hi], "edges": [float(edges[b]), float(edges[b + 1])]})
    n = len(p)
    return {"brier": float(np.mean((p - y) ** 2)),
            "reliability": rel / n, "resolution": res / n,
            "uncertainty": float(base * (1 - base)), "base_rate": float(base),
            "curve": curve}


BR = brier_decomposition(q0_test, is0_test)
print(f"\nBrier {BR['brier']:.4f} = reliability {BR['reliability']:.4f} "
      f"- resolution {BR['resolution']:.4f} + uncertainty {BR['uncertainty']:.4f}")
print("  reliability -> 0 is a correct SCALE; resolution -> large is a useful RANKING.")
print(f"\nreliability diagram of q(0|x)  (base rate {BR['base_rate']:.3f}):")
print(f"{'q(0|x) decile':>20}{'n':>8}{'mean q':>10}{'observed':>10}{'95% Wilson':>18}")
for c in BR["curve"]:
    print(f"{c['edges'][0]:>9.4f}-{c['edges'][1]:<10.4f}{c['n']:>8}"
          f"{c['p_mean']:>10.3f}{c['observed']:>10.3f}"
          f"   [{c['ci'][0]:.3f}, {c['ci'][1]:.3f}]")
print("  `observed` systematically ABOVE `mean q` is the under-confidence "
      "PLAN_empty_parton_tree.md F5\n  measures at ~2x — and SBC/PIT cannot see it, "
      "because SBC ranks against the sampler's own draws.")


In [ ]:
# --- 3. recalibration: fitted on the TRAINING val split, applied FROZEN ------
T_only, _ = fit_length_recalibration(pmf_fit, n_fit, with_tilt=False)
T_fit, TILT_fit = fit_length_recalibration(pmf_fit, n_fit, with_tilt=True)
print(f"fitted on {len(pmf_fit)} training-val jets:")
print(f"  scalar temperature only : T = {T_only:.4f}   (the plan's original proposal)")
print(f"  temperature + tilt      : T = {T_fit:.4f}, tilt = {TILT_fit:+.4f}")


def length_report(pmfs, n_true, T=1.0, tilt=0.0):
    P = [recalibrate_pmf(p, T, tilt) for p in pmfs]
    q0 = np.array([p[0] for p in P])
    nll = float(np.mean([-math.log(max(p[n], 1e-300)) if n < len(p) else np.nan
                         for p, n in zip(P, n_true)]))
    return {"mean_q0": float(q0.mean()), "truth_rate": float((n_true == 0).mean()),
            "ratio": float((n_true == 0).mean() / max(q0.mean(), 1e-12)),
            "nll_of_N": nll, "auc": auc(q0, n_true == 0),
            "brier": brier_decomposition(q0, n_true == 0)}


ROWS = [("uncalibrated", 1.0, 0.0),
        ("temperature only", T_only, 0.0),
        ("temperature + tilt", T_fit, TILT_fit)]
print(f"\n{'variant':<20}{'mean q(0|x)':>13}{'truth':>8}{'emp/pred':>10}"
       f"{'NLL of N':>10}{'AUC':>7}{'Brier':>8}{'reliab.':>9}")
RECAL = {"fit_jets": len(pmf_fit), "T_only": T_only, "T": T_fit, "tilt": TILT_fit,
         "variants": {}}
for label, T_var, tilt in ROWS:
    # not `T`: that name holds section 2b's per-term NLL arrays, and shadowing it makes
    # a re-run of 2b's summary cell fail on a float
    r = length_report(pmf_test, n_test, T_var, tilt)
    print(f"{label:<20}{r['mean_q0']:>13.4f}{r['truth_rate']:>8.4f}{r['ratio']:>10.3f}"
          f"{r['nll_of_N']:>10.4f}{r['auc']:>7.3f}{r['brier']['brier']:>8.4f}"
          f"{r['brier']['reliability']:>9.4f}")
    RECAL["variants"][label] = {k: v for k, v in r.items() if k != "brier"}
    RECAL["variants"][label]["brier"] = {k: v for k, v in r["brier"].items()
                                         if k != "curve"}
print("\nA scalar temperature CANNOT close this gap: it is symmetric about the mode, so "
      "it pulls\nq(0|x) DOWN toward 1/(max_emissions+1). The measured error is a "
      "monotone ramp across n,\nwhich needs the tilt. The AUC barely moves under either "
      "— recalibration fixes the SCALE,\nnot the ranking, which is exactly what the "
      "reliability term above says is broken.")
RECAL["reliability_curve_uncalibrated"] = BR["curve"]


In [ ]:
# --- 4. the gate: tau fitted on the training val split, applied frozen -------
# The truth empty rate on the FIT set is what tau is matched to. Applying it to the test
# file then MEASURES whether the rate transfers, instead of reproducing it by
# construction.
#
# **tau must be fitted on the SCALE IT WILL BE APPLIED TO.** `empty_gate` thresholds
# `q(0|x)`, and `empty_threshold_for_rate` returns a QUANTILE of it — scale-free in its
# ranking, but a specific number on a specific distribution. The deployed decode carries
# `(T, tilt)`, so the `q(0|x)` the gate meets in production is the RECALIBRATED one,
# whose mean is ~3x the raw head's. Fitting on raw and applying to recalibrated is the
# same class of error `length.py`'s docstring warns about for a selection change, and it
# is not subtle: the raw-scale tau cuts the top ~16% of the raw distribution and the top
# ~45% of the recalibrated one. Both are computed below so the mismatch is visible
# rather than merely avoided.
rate_fit = float((n_fit == 0).mean())
pmf_fit_cal = [recalibrate_pmf(p, T_fit, TILT_fit) for p in pmf_fit]
pmf_test_cal = [recalibrate_pmf(p, T_fit, TILT_fit) for p in pmf_test]

TAU = empty_threshold_for_rate(pmf_fit_cal, rate_fit)      # the deployed scale
TAU_RAW = empty_threshold_for_rate(pmf_fit, rate_fit)      # kept only to price the bug
truth0 = n_test == 0


def gate_scores(pmfs, tau):
    fired = np.array([empty_gate(p, tau) for p in pmfs])
    tp = int((fired & truth0).sum())
    return {"p_empty_pred": float(fired.mean()),
            "ratio": float(fired.mean()) / max(float(truth0.mean()), 1e-9),
            "recall": tp / max(int(truth0.sum()), 1),
            "precision": tp / max(int(fired.sum()), 1)}


GATE = gate_scores(pmf_test_cal, TAU)
GATE_MISMATCH = gate_scores(pmf_test_cal, TAU_RAW)
fired = np.array([empty_gate(p, TAU) for p in pmf_test_cal])
recall, precision = GATE["recall"], GATE["precision"]
print(f"tau = {TAU:.5f}   fitted on the RECALIBRATED q(0|x) — the scale the deployed")
print(f"      decode presents — to reproduce the training-val empty rate {rate_fit:.4f}")
print(f"applied FROZEN to the test file:")
print(f"  p_empty_true = {float(truth0.mean()):.4f}   p_empty_pred = "
      f"{GATE['p_empty_pred']:.4f}   (ratio {GATE['ratio']:.3f})")
print(f"  recall = {recall:.3f}   precision = {precision:.3f}")
print(f"\n  the scale mismatch, priced: tau fitted on the RAW head ({TAU_RAW:.5f}) and")
print(f"  applied to the same recalibrated q(0|x) gives rate "
      f"{GATE_MISMATCH['p_empty_pred']:.4f} ({GATE_MISMATCH['ratio']:.2f}x),")
print(f"  recall {GATE_MISMATCH['recall']:.3f}, precision {GATE_MISMATCH['precision']:.3f}"
      f" — the ranking is identical, only the cut moved.")
print(f"  MBR backend for the point estimate: {MBR_BACKEND}  — this is the ONE "
      f"observable that\n  swings with the backend (pot ~0.2%, surrogate ~57%); no "
      f"other panel does.")

# --- validate p_empty_true independently (check 9) --------------------------
# `run_closure` computes its empty-tree keys BEFORE the `ly is None` continue. If that
# ordering ever regresses, `p_empty_true` reads exactly 0.0 — so compare it against a
# rate computed straight off the truth here, and assert.
p_true_direct = float(np.mean([len(jets_samp[k]["y"][0]) == 0 for k in range(len(jets_samp))]))
assert abs(CLO["p_empty_true"] - p_true_direct) < 1e-9, (
    f"run_closure p_empty_true={CLO['p_empty_true']} disagrees with the direct truth "
    f"rate {p_true_direct}: the empty-tree accounting has moved back BEHIND the "
    f"leading-emission `continue`, and every empty-tree number is conditioned on "
    f"non-empty truth."
)
assert CLO["p_empty_true"] > 0.0, "p_empty_true == 0.0 exactly is the tell"
print(f"\nrun_closure p_empty_true = {CLO['p_empty_true']:.4f} matches the direct truth "
      f"rate {p_true_direct:.4f}\n(computed before the leading-emission selection, as it "
      f"must be).")

METRICS["empty_tree"] = {
    "auc_q0": AUC, "brier": {k: v for k, v in BR.items() if k != "curve"},
    "reliability_curve": BR["curve"],
    "recalibration": RECAL,
    "tau": {"value": float(TAU), "fitted_on": "training-file val split",
            "fit_rate": rate_fit, "n_fit_jets": len(pmf_fit),
            "p_empty_true": float(truth0.mean()), **GATE,
            "mbr_backend": MBR_BACKEND,
            # tau is a QUANTILE of q(0|x), so it is only meaningful on the scale it was
            # fitted on. Recording that scale is what lets a consumer assert it applies
            # the same one — `lund_distribution_closure_prod_test_v1.ipynb` does.
            "fitted_under": {"length_temperature": float(T_fit),
                             "length_tilt": float(TILT_fit)},
            # what the same tau does on the WRONG scale, kept so the failure has a price
            "raw_scale_value": float(TAU_RAW),
            "raw_scale_applied_to_recalibrated": GATE_MISMATCH},
    "run_closure_p_empty_true": CLO["p_empty_true"],
    "direct_truth_rate": p_true_direct,
}


## 7. Distribution closure — a pointer, deliberately not a re-implementation

[`lund_distribution_closure_v2.ipynb`](lund_distribution_closure_v2.ipynb) already
computes the population-level W1/KS/χ² improvement ratios with bootstrap noise floors
and scoreability gating. Duplicating that here would create a **second definition of
the headline number**, and this repo has already been burned by two closure
populations drifting apart.

**Do not hand-edit v2 for this.** Run
[`lund_distribution_closure_prod_test_v1.ipynb`](lund_distribution_closure_prod_test_v1.ipynb),
which is v2 with every setting already applied — generated by
[`scripts/make_prod_closure_nb.py`](../scripts/make_prod_closure_nb.py), byte-identical
to v2 outside its title and section 0, and reading the five settings from the
`prod_test_v1_metrics.json` this notebook writes in §9. Nothing to copy, nothing to
forget, and no second copy of the analysis to drift.

The cell below prints those settings anyway, because the generated notebook resolves
them at runtime and you should be able to see what it will pick up. **Three of them are
not defaults you could have left alone.** The `(T, tilt)` fitted in §6 reaches `sample`,
so it moves closure_v2's posterior series *and* its empty rate — an artifact that does
not record them cannot have its empty rate attributed. And its `EMPTY_THRESHOLD` default
(`None`) rate-matches tau on the sample it reports on, reproducing its own fitted rate
by construction; the frozen tau from §6 is what makes it a measurement.


In [ ]:
# Everything closure_v2 needs for THIS run, printed rather than written down so it cannot
# drift from the checkpoint actually loaded above.
_rows = [
    ("CKPT_PATH", f'"{rel(CKPT)}"', "CHANGE",
     "this run's checkpoint (section 1)"),
    ("ROOT_PATH", f'"{ROOT_PATH}"', "CHANGE",
     "the independent test file; its default is cpp/test_data/jets.root"),
    ("EMPTY_THRESHOLD", f"{TAU:.5f}", "CHANGE",
     "the tau FROZEN from the training val split (section 6). Its default None "
     "rate-matches on the sample it reports on, which is circular"),
    ("LENGTH_TEMPERATURE", f"{T_fit:.4f}", "CHANGE",
     "fitted on the training val split (section 6); reaches length_pmf AND sample"),
    ("LENGTH_TILT", f"{TILT_fit:+.4f}", "CHANGE",
     "the same fit. A scalar temperature alone cannot correct this head"),
    ("PLANE_NB", f"{geom.n_bins}", "already right",
     f"documented as a multiple of geometry.n_bins; at n_bins={geom.n_bins} that is 1x, "
     f"so the plane shows the model's own cell granularity. {2 * geom.n_bins} only for "
     f"sub-cell resolution"),
    ("MBR_BACKEND", '"energyflow" (falls back to "pot")', "set by the generator",
     "the SAME perturbative-Lund EMD as pot — identical MBR tree on 99.3% of jets, 100% "
     "on multiplicity, the rest solver tie-breaks — and ~3x faster, which is most of "
     "that notebook's speedup (docs/PLAN_prod_test_speedup.md Part B). surrogate is a "
     "different risk function and stays forbidden"),
    ("REQUIRE_TRUTH_SPLITTING", "False", "already right",
     "the deployable population; selecting on len(y)>0 reads the answer"),
]


def _wrap(text, width, indent):
    out, line = [], ""
    for w in text.split():
        if len(line) + len(w) + 1 > width:
            out.append(line)
            line = w
        else:
            line = f"{line} {w}".strip()
    out.append(line)
    return ("\n" + " " * indent).join(out)


print("lund_distribution_closure_prod_test_v1.ipynb resolves these from the artifact")
print("written in section 9 below — this is what it will pick up:\n")
for _name, _val, _status, _why in _rows:
    # value on its own line: a checkpoint path does not fit a column, and truncating the
    # one thing you have to copy verbatim would be the worst possible economy
    print(f"  {_name:<23} [{_status}]")
    print(f"      = {_val}")
    print(f"        {_wrap(_why, 72, 8)}")
print("\nEverything else in its section 0 is already correct for this run — N_JETS, SEED,")
print("DEVICE, K_DRAWS, the rebinning and the scoreability gate all keep their defaults,")
print("and the generated notebook ASSERTS the rows it does not set rather than trusting")
print("them, so a future change to a v2 default fails instead of drifting.")
print("\nLENGTH_TEMPERATURE / LENGTH_TILT did not exist as knobs until this run needed")
print("them: closure_v2 read them from the checkpoint snapshot, so it could only ever")
print("record (1.0, 0.0) — the identity — however they had been fitted.")

# Deliberately NOT written into METRICS: every one of these already has a primary record
# in this file (`run.checkpoint`, `run.test_path`, `empty_tree.tau.value`,
# `empty_tree.recalibration.T` / `.tilt`), and a summary block duplicating them is a
# second copy that can go stale. The generated notebook reads the primaries.


In [ ]:
# Staleness guard: refuse to quote a dist_closure_metrics.json that describes a
# different run. A stale artifact beside the right checkpoint is worse than none.
dc = CKPT.parent / "dist_closure_metrics.json"
DIST = None
if dc.exists():
    j = json.loads(dc.read_text())
    got = {"data.path": j.get("data", {}).get("path", j.get("root_path")),
           "checkpoint": j.get("checkpoint"),
           # closure_v2 nests this under `data`, not at the top level; reading the wrong
           # path printed a confident `None` beside two fields that had matched
           "n_eval_jets": j.get("data", {}).get("n_eval_jets", j.get("n_eval_jets"))}
    same_file = str(got["data.path"]).endswith(Path(ROOT_PATH).name)
    # the RUN directory, not just "best.ckpt" — every run writes a file by that name, so
    # matching on the basename alone would accept an artifact from any other run
    same_ckpt = (Path(str(got["checkpoint"])).parent.name == CKPT.parent.name
                 if got["checkpoint"] else False)
    used_T = j.get("decode", {}).get("length_temperature", 1.0)
    used_tilt = j.get("decode", {}).get("length_tilt", 0.0)
    if same_file and same_ckpt:
        DIST = j
        print(f"dist_closure_metrics.json accepted: {got}")
        print(f"  its posterior series was drawn at length_temperature={used_T}, "
              f"length_tilt={used_tilt}.")
        if abs(float(used_T) - T_fit) > 1e-6 or abs(float(used_tilt) - TILT_fit) > 1e-6:
            print(f"  NOTE: that is NOT the ({T_fit:.4f}, {TILT_fit:+.4f}) fitted in "
                  f"section 6. Recalibration reaches\n  `sample`, so its empty rate "
                  f"describes the UNCALIBRATED head. Both are legitimate — but they "
                  f"are\n  different numbers, and this is what makes that attributable "
                  f"rather than ambiguous.")
    else:
        print(f"REFUSING to quote {dc.name}: it describes {got}, not this run "
              f"({ROOT_PATH}, {rel(CKPT)}).\nRe-run closure_v2 with the constants above.")
else:
    print(f"no dist_closure_metrics.json beside the checkpoint yet — section 7 is "
          f"a pointer\nuntil closure_v2 has been run on this file.")
METRICS["distribution_closure"] = {
    "artifact": str(rel(dc)) if dc.exists() else None,
    "accepted": DIST is not None,
    "fitted_length_temperature": T_fit, "fitted_length_tilt": TILT_fit,
    "artifact_length_temperature": (float(used_T) if DIST else None),
    "artifact_length_tilt": (float(used_tilt) if DIST else None),
    "headline": (DIST or {}).get("headline"),
}


## 8. Support and validity — what the model can and cannot produce

The same three fractions as
[`lund_distribution_closure_v2.ipynb`](lund_distribution_closure_v2.ipynb) §5, on the
same definitions: emissions outside the fiducial Lund window, emissions violating the
Soft Drop condition the file was groomed with, and emissions below the `k_t` floor.
The truth's own rates are the **irreducible floor** — a posterior cannot be faulted
for a violation the target sample also shows.


In [ ]:
U_LO, U_HI = geom.ln_invdelta_range
V_LO, V_HI = geom.ln_kt_range
Z_CUT, BETA = float(prov["z_cut"]), float(prov["beta"])
LNZ_FLOOR = math.log(Z_CUT)


def support_row(v):
    """`v` is (n, 4): ln 1/DeltaR, ln kt, ln z, psi — the `node_raw` layout."""
    v = np.asarray(v, dtype=float)
    if not len(v):
        return dict(n=0, out_of_window=float("nan"), sd_violation=float("nan"),
                    ktfloor_violation=float("nan"))
    oow = ((v[:, 0] < U_LO) | (v[:, 0] > U_HI) | (v[:, 1] < V_LO) | (v[:, 1] > V_HI))
    sd = v[:, 2] <= (LNZ_FLOOR - BETA * v[:, 0])
    return dict(n=int(len(v)), out_of_window=float(oow.mean()),
                sd_violation=float(sd.mean()),
                ktfloor_violation=float((v[:, 1] < V_LO).mean()))


# The truth and identity rows cost no model call, so they are measured on the WHOLE
# population — they are the floor every other row is read against, and a floor measured
# on 300 jets is not a floor. Only the posterior row needs the HEAVY tier.
pool_truth = [node_raw(*jets[i]["y"]) for i in POP if len(jets[i]["y"][0])]
pool_x = [node_raw(*jets[i]["x"]) for i in POP if len(jets[i]["x"][0])]

# A SLICE of §1's draws (v0 re-sampled the HEAVY tier here at K_DRAWS // 10), and one
# batched coordinate call per jet instead of one per draw: 4.1 min -> ~7 s.
K_SUPPORT = max(K_DRAWS // 10, 1)
seed_everything(SEED)
pool_post = []
with torch.inference_mode():
    for k in range(len(ds_heavy)):
        item = ds_heavy[k]
        xf = item["xf"].unsqueeze(0).to(device)
        nx = torch.tensor([item["nx"]], device=device)
        for c in model.sample_coordinates_many(
                xf, nx, [list(d) for d in DRAWS[k][:K_SUPPORT] if len(d)]):
            if c is not None:
                pool_post.append(c.detach().cpu().double().numpy().reshape(-1, 4))

SERIES = {"truth y": pool_truth, "identity x": pool_x, "posterior": pool_post}
print(f"truth/identity measured on all {len(POP)} jets; posterior on the "
      f"{len(ds_heavy)}-jet HEAVY tier ({K_SUPPORT} of §1's draws per jet).")
SUPPORT = {s: support_row(np.concatenate(v) if v else np.zeros((0, 4)))
           for s, v in SERIES.items()}
print(f"{'series':<12}{'emissions':>11}{'out of window':>15}{'soft-drop viol.':>17}"
      f"{'kt-floor viol.':>16}")
print("-" * 71)
for s, r in SUPPORT.items():
    def f(x):
        return "     n/a" if not np.isfinite(x) else f"{100 * x:7.3f}%"
    print(f"{s:<12}{r['n']:>11}{f(r['out_of_window']):>15}{f(r['sd_violation']):>17}"
          f"{f(r['ktfloor_violation']):>16}")
print(f"\nfiducial window: ln(1/dR) in [{U_LO}, {U_HI}], ln kt in [{V_LO}, {V_HI}], "
      f"ln z > {LNZ_FLOOR:.3f} - {BETA:g}*ln(1/dR)")
print("the truth row is the irreducible floor: a posterior is not at fault for a "
      "violation the\ntarget sample also shows. Only the EXCESS over it is the model's.")
METRICS["support"] = SUPPORT


## 9. Summary

The acceptance criterion, per
[`PLAN_ProductionAssessment.md`](../docs/PLAN_ProductionAssessment.md) §9 as amended by
`PLAN_prod_test_v0.md`: the arm must beat identity on **`dlund_posterior_medoid`** *and*
on **`dlund_posterior_geomedian_cont`**. If the two disagree in sign the arm is
quantisation-limited, not better or worse.


In [ ]:
med_ratio = CLO["dlund_posterior_medoid"] / CLO["dlund_identity"]
cont_ratio = (CLO_C["dlund_posterior_geomedian_cont"] / CLO_C["dlund_identity_cont"]
              if np.isfinite(CLO_C.get("dlund_posterior_geomedian_cont", np.nan))
              else float("nan"))
beats_cell = med_ratio < 1.0
beats_cont = cont_ratio < 1.0
if beats_cell and beats_cont:
    ACCEPT = "PASS — beats identity on both the cell medoid and the off-grid geo-median"
elif beats_cell != beats_cont:
    ACCEPT = ("SPLIT — the two estimators DISAGREE IN SIGN, so the arm is "
              "quantisation-limited, not better or worse")
else:
    ACCEPT = "FAIL — does not beat identity on either estimator"

print(f"{'':<44}{'value':>10}{'vs identity':>13}")
print(f"{'dlund_posterior_medoid (cell)':<44}{CLO['dlund_posterior_medoid']:>10.4f}"
      f"{med_ratio:>13.3f}")
print(f"{'dlund_posterior_geomedian_cont (off grid)':<44}"
      f"{CLO_C.get('dlund_posterior_geomedian_cont', float('nan')):>10.4f}"
      f"{cont_ratio:>13.3f}")
print(f"\nACCEPTANCE: {ACCEPT}")

print(f"\n{'-' * 78}\nheadline numbers, with what qualifies each\n{'-' * 78}")
summary = [
    ("held-out NLL/jet (whole test file)", f"{METRICS['nll']['total_per_jet']:.4f}",
     "density on the plane; comparable across n_bins, but a finer grid is a richer class"),
    ("split term, per emission", f"{METRICS['nll']['split_per_emission']:.4f}",
     f"NOT comparable to 10 bins (+{METRICS['nll']['split_ll_shift_vs_10_bins']:.3f})"),
    ("aux ON - aux OFF", f"{METRICS['aux_ablation'].get('delta_nat_per_jet', float('nan')):+.4f}",
     METRICS["aux_ablation"].get("verdict", "no seed band -> not rankable")),
    ("leading-emission medoid / identity", f"{med_ratio:.3f}",
     f"p(leading | n_y > 0), {CLO['n_kept_leading']}/{CLO['n_jets_scored']} jets kept"),
    ("off-grid geo-median / identity", f"{cont_ratio:.3f}",
     f"{CLO_C.get('n_continuous_jets', 0)} jets"),
    ("leading-cell 68% coverage", f"{CAL['coverage_68']:.3f}",
     f"95% Wilson [{CAL['coverage_68_ci'][0]:.3f}, {CAL['coverage_68_ci'][1]:.3f}] on "
     f"{CAL['n_coverage']} jets"),
    ("SBC chi2 (uniformity of the N rank)", f"{CAL['sbc_chi2_uniform']:.2f}",
     f"vs chi2({CAL['sbc_chi2_dof']}) 95% point {CAL['sbc_chi2_crit95']:.2f}"),
    ("TARP max |ECP - alpha|", f"{TARP['tarp_max_dev']:.3f}",
     f"vs null floor {TARP['tarp_null_floor95']:.3f} at {TARP['n_jets']} jets"),
    ("q(0|x) AUC", f"{AUC:.3f}", "ranking only; the SCALE is the reliability term"),
    ("Brier reliability, uncalibrated",
     f"{METRICS['empty_tree']['brier']['reliability']:.4f}",
     f"-> {RECAL['variants']['temperature + tilt']['brier']['reliability']:.4f} with "
     f"(T={T_fit:.3f}, tilt={TILT_fit:+.3f})"),
    ("gated empty rate (tau frozen from train val)",
     f"{METRICS['empty_tree']['tau']['p_empty_pred']:.4f}",
     f"truth {METRICS['empty_tree']['tau']['p_empty_true']:.4f}, "
     f"recall {METRICS['empty_tree']['tau']['recall']:.3f}, MBR backend {MBR_BACKEND}"),
    ("cells the posterior misses",
     f"{METRICS['occupancy']['truth_cells_missed_by_posterior']}",
     f"of {METRICS['occupancy']['truth_cells_occupied']} truth-occupied; split_head "
     f"effective rank {METRICS['occupancy']['effective_rank']:.1f}/"
     f"{METRICS['occupancy']['rank_bound']}"),
]
for name, val, note in summary:
    print(f"{name:<44}{val:>10}   {note}")

METRICS["acceptance"] = {
    "dlund_posterior_medoid": CLO["dlund_posterior_medoid"],
    "dlund_identity": CLO["dlund_identity"],
    "medoid_ratio": med_ratio,
    "dlund_posterior_geomedian_cont": CLO_C.get("dlund_posterior_geomedian_cont"),
    "dlund_identity_cont": CLO_C.get("dlund_identity_cont"),
    "geomedian_cont_ratio": cont_ratio,
    "beats_identity_cell": bool(beats_cell), "beats_identity_cont": bool(beats_cont),
    "verdict": ACCEPT,
}
# check 11: which decode knobs never reached any number above.
METRICS["decode"] = dict(DECODE)
METRICS["decode_inert"] = inert_decode_keys(model, dict(DECODE))
print(f"\ndecode knobs that did NOT reach any number above ({len(METRICS['decode_inert'])}):")
for e in METRICS["decode_inert"]:
    print(f"    {e['key']} = {e['value']!r}   — {e['reason']}")


In [ ]:
# v1 changed the RNG regime, not the assessment — so if a v0 artifact sits beside this
# checkpoint, the headline numbers belong side by side. Nothing here is asserted: the two
# runs are NOT bit-comparable by construction (shared draws + batched coordinate heads
# reorder RNG consumption), and a Monte-Carlo difference is not a failure. What would be
# a failure is a difference too big to be one, and the bands to read it against are on
# hand: the Wilson interval for `coverage_68`, and the four PIT chunks' KS spread (§4)
# as the general MC scale at these tiers.
_v0_path = CKPT.parent / "prod_test_v0" / "prod_test_v0_metrics.json"
if _v0_path.exists():
    V0 = json.loads(_v0_path.read_text())

    def _g(d, *keys):
        for k in keys:
            if not isinstance(d, dict) or k not in d:
                return float("nan")
            d = d[k]
        return float(d) if isinstance(d, (int, float)) else float("nan")

    _cmp_rows = [
        ("dlund_identity (cell)", _g(V0, "closure", "cell_tier", "dlund_identity"),
         CLO["dlund_identity"]),
        ("dlund_posterior_mode", _g(V0, "closure", "cell_tier", "dlund_posterior_mode"),
         CLO["dlund_posterior_mode"]),
        ("dlund_posterior_medoid", _g(V0, "closure", "cell_tier", "dlund_posterior_medoid"),
         CLO["dlund_posterior_medoid"]),
        ("medoid / identity", _g(V0, "acceptance", "medoid_ratio"), med_ratio),
        ("dlund_geomedian_cont", _g(V0, "closure", "continuous_tier",
                                    "dlund_posterior_geomedian_cont"),
         CLO_C.get("dlund_posterior_geomedian_cont", float("nan"))),
        ("geo-median / identity (off grid)", _g(V0, "acceptance", "geomedian_cont_ratio"),
         cont_ratio),
        ("coverage_68", _g(V0, "calibration", "coverage_68"), CAL["coverage_68"]),
        ("SBC chi2", _g(V0, "calibration", "sbc_chi2_uniform"), CAL["sbc_chi2_uniform"]),
        ("mult_bias_posterior", _g(V0, "closure", "cell_tier", "mult_bias_posterior"),
         CLO["mult_bias_posterior"]),
        ("collect(): cell medoid ratio", _g(V0, "closure", "leading", "cell_medoid", "ratio"),
         LEAD["cell_medoid"]["ratio"]),
        ("posterior cells emitted", _g(V0, "occupancy", "posterior_cells_emitted"),
         float(METRICS["occupancy"]["posterior_cells_emitted"])),
        ("TARP max |ECP - alpha|", _g(V0, "calibration", "tarp", "tarp_max_dev"),
         TARP["tarp_max_dev"]),
    ]
    print(f"\nv0 (independent draws per section) vs v1 (one shared pass), same "
          f"checkpoint and tiers:")
    print(f"{'quantity':<34}{'v0':>10}{'v1':>10}{'delta':>10}{'rel':>9}")
    for _n, _a, _b in _cmp_rows:
        _rel = (_b - _a) / _a if _a else float("nan")
        print(f"{_n:<34}{_a:>10.4f}{_b:>10.4f}{_b - _a:>+10.4f}{_rel:>+9.2%}")
    _ci = CAL["coverage_68_ci"]
    _v0ci = V0.get("calibration", {}).get("coverage_68_ci", [float("nan")] * 2)
    print(f"\n  coverage_68 95% Wilson: v0 [{_v0ci[0]:.3f}, {_v0ci[1]:.3f}]   "
          f"v1 [{_ci[0]:.3f}, {_ci[1]:.3f}]   "
          f"{'OVERLAP' if _ci[0] <= _v0ci[1] and _v0ci[0] <= _ci[1] else 'DISJOINT — look'}")
    print("  the `*_cont` rows move most, and are expected to: they are the ones whose "
          "draws\n  now come from `sample_coordinates_many` rather than K separate calls.")
    print(f"  v0 artifact: {rel(_v0_path)}")
else:
    print(f"\n(no prod_test_v0 artifact beside this checkpoint — nothing to compare "
          f"against;\n expected at {rel(CKPT.parent / 'prod_test_v0')})")


In [ ]:
# `plot_calibration` does `matplotlib.use("Agg")` INSIDE it (eval/report.py), which in a
# live kernel switches the backend for the rest of the session and renders every later
# `plt.show()` blank. It is therefore the LAST figure call in this notebook, deliberately.
#
# Artifacts go to `<ckpt dir>/prod_test_v1/`, not `<ckpt dir>/`: `plot_calibration`
# writes FIXED filenames that would otherwise clobber what `h2p-rsd-junipr eval` wrote.
if WRITE_ARTIFACTS:
    # `prod_test_v1/`, beside — not over — v0's: the two artifacts are the same
    # assessment under different RNG regimes and the comparison above needs both.
    OUT = CKPT.parent / "prod_test_v1"
    OUT.mkdir(parents=True, exist_ok=True)
    figs = plot_calibration({**CAL, "pit_coords": PITS[0] if PITS else None,
                             "tarp": TARP}, OUT)
    p = save_metrics(METRICS, OUT / "prod_test_v1_metrics.json")
    print(f"wrote {p}")
    for f in figs:
        print(f"wrote {f}")
else:
    print("WRITE_ARTIFACTS is False — nothing written.")
